# 第 1 周末练习 —— 技术问答解释器（OpenAI 流式）

## 练习目标（理念）

为展示你对 **OpenAI API**（以及可选的本地 **Ollama**）的熟悉程度，请构建一个小工具：

- **输入**：一个技术问题（例如「这段 Python 代码在干什么？」）
- **输出**：清晰、严谨的解释（以 Markdown 展示）
- **额外要求**：用**流式（streaming）**一边生成一边刷新显示，而不是等整段答完才一次性打印

这是你在课程期间自己也能天天用的工具：遇到看不懂的代码，丢进来问模型。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `openai.chat.completions.create(...)` |
| `messages`（system / user） | system 定「怎么答」，user 放具体问题 |
| 流式输出 `stream=True` | 逐块拼接并用 `update_display` 刷新 |
| OpenAI 云端模型 | `gpt-5-nano`（常量 `MODEL_GPT`） |
| Ollama 本地模型 | 本笔记本最后一格预留了 Llama 入口，当前未实现调用 |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env`：至少有 `OPENAI_API_KEY`（本练习用默认 `OpenAI()` 客户端读取）
3. 在「提问」单元格改写 `question`，再跑提示词与流式回答两格
4. 若要接本地 Llama，可在最后一格自行补全（需本机 Ollama 与 `llama3.2`）


In [1]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：Markdown 渲染、初次 display、流式 update_display
from IPython.display import Markdown, display, update_display
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI


> （本格原为空）下面进入「常量 / 环境 / 提问 / 调用」主流程。  
> 理念：先集中写模型名与密钥加载，再拼 `messages`，最后用 `stream=True` 边收边显示。


In [2]:
# ========== 常量：模型名字集中写在一处，后面只改这里 ==========

# OpenAI 云端小模型：比 gpt-4o-mini 更便宜；与 day1/day5 常用名一致
MODEL_GPT = 'gpt-5-nano'   # cheaper than gpt-4o-mini; same as day1/day5
# 本地 Ollama 模型名预留：作者注明本练习暂不走 Ollama，故整行注释掉
# MODEL_LLAMA = 'llama3.2' # 不使用 Ollama


In [3]:
# ========== 环境：加载密钥并创建 OpenAI 客户端 ==========

# load_dotenv(override=True)：读取 .env；override=True 表示用文件值覆盖已有环境变量
load_dotenv(override=True)
# 创建默认 OpenAI 客户端：会从环境变量 OPENAI_API_KEY 取密钥
openai = OpenAI()


In [4]:
# ========== 提问：改这里的字符串就能问新问题 ==========

# 把技术问题写在三引号字符串里；发给模型的内容保持英文（可运行 / 影响回答的字符串不翻译）
# 练习建议：换成你自己今天看不懂的一行代码，再跑后面的提示词与流式回答格
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""


In [5]:
# ========== 提示词 + messages：告诉模型「你是谁」和「要答什么」 ==========

# system prompt 保留英文：这是发给模型的角色指令，改译可能改变回答风格/行为
system_prompt = "You are a helpful technical tutor who answers questions about python code, software engineering, data science and LLMs."
# user prompt：在固定前缀后拼上具体 question（字符串拼接，保持原逻辑）
user_prompt = "Please give a detailed explanation to the following question: " + question

# messages：Chat Completions 的标准对话列表；顺序一般是 system → user
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]


In [6]:
# ========== 路径 A：用云端 MODEL_GPT（gpt-5-nano）流式回答 ==========
# 注释里仍写 gpt-4o-mini 是原作者习惯；实际 model= 用的是上面的 MODEL_GPT

# stream=True：不要等整段生成完，而是持续返回增量 delta
stream = openai.chat.completions.create(model=MODEL_GPT, messages=messages, stream=True)

# response：累积已收到的全部文本，供每次刷新 Markdown 使用
response = ""
# display_id=True：拿到可更新的显示句柄，后面用 update_display 原地刷新
display_handle = display(Markdown(""), display_id=True)
# 遍历流式 chunk：每一小块可能带一点新 content
for chunk in stream:
    # delta.content 可能为 None（例如角色/结束事件），用 or '' 避免把 None 拼进去
    response += chunk.choices[0].delta.content or ''
    # 用累积后的完整 Markdown 刷新同一个 display 区域，实现「打字机」效果
    update_display(Markdown(response), display_id=display_handle.display_id)


Short answer
- It collects all non-empty author values from a sequence of dicts (books) into a set (which removes duplicates), and then yields each author from that set one by one to the caller.

What each part does
- books is assumed to be an iterable of dictionaries, each representing a book.
- book.get("author"):
  - Retrieves the value for the key "author" from a book dict.
  - If the key is missing, get returns None.
- if book.get("author"):
  - This filters out any books whose author is missing or evaluates to a falsy value (None, empty string, etc.).
- {book.get("author") for book in books if book.get("author")}:
  - This is a set comprehension. It builds a set of all author values (one per book) that pass the filter.
  - Using a set automatically deduplicates authors, so duplicates are removed.
  - Sets are unordered, so the order of authors in the resulting set is not defined.
- yield from { ... }:
  - yield from delegates to the iterable (here, the set of authors) and yields each element to the caller.
  - In effect, the code emits each unique author exactly once.

Why you might use this
- You want to lazily produce a sequence of unique authors from a collection of books.
- The use of a set ensures each author is emitted only once (no duplicates).
- yield from lets the function be a generator and stream authors to the caller, rather than returning a list of authors.

Important nuances and potential improvements
- Order is not guaranteed because sets are unordered. If you care about the order of first appearance, this approach won’t preserve it.
  Example of preserving order while deduplicating:
  - def iter_authors_ordered(books):
      seen = set()
      for book in books:
          author = book.get("author")
          if author and author not in seen:
              seen.add(author)
              yield author
- If you still want to use a set but preserve insertion order, you could collect authors with a dict (Python 3.7+ dicts preserve insertion order):
  - yield from dict.fromkeys(book.get("author") for book in books if book.get("author"))
  - This yields authors in the order they first appear, without duplicates.

Example
- Given:
  books = [
    {"title": "A", "author": "Alice"},
    {"title": "B", "author": "Bob"},
    {"title": "C", "author": "Alice"},
    {"title": "D"}  # no author
  ]
- The code yields: "Alice" and "Bob" (order is not guaranteed due to set), one time each.

In [ ]:
# ========== 路径 B（预留）：让 Llama 3.2 来回答 ==========
# 本格目前只有标题注释，没有可执行调用 —— 逻辑保持原样，不擅自补代码
# 若要练习 Ollama：可新建 OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# 再用 model="llama3.2"、同一套 messages 调 chat.completions.create（可选 stream=True）
# 让 Llama 3.2 来回答
